# 03 - Speed Safety Score, Risk Tiers, and Sensitivity Analysis

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import geopandas as gpd

from src.score import BASE_WEIGHTS, score_and_classify, run_sensitivity_analysis, INSUFFICIENT_DATA_TIER

pd.set_option('display.max_columns', None)

reliable = gpd.read_parquet('../data/processed/_reliable_features.parquet')
reliable = gpd.GeoDataFrame(reliable, geometry='geometry', crs='EPSG:4326')
low_confidence = gpd.read_parquet('../data/processed/_low_confidence.parquet')
low_confidence = gpd.GeoDataFrame(low_confidence, geometry='geometry', crs='EPSG:4326')

## Score and classify

Weights: `speed_gap_norm` 0.30, `road_mismatch` 0.25, `bio_risk` 0.30, `vru_exposure` 0.15. Weighted sum is multiplied by `confidence_weight` and 100. Tiers: High >= 70, Medium >= 40, Low < 40; low-confidence segments are labelled 'Insufficient data'.

In [2]:
reliable = score_and_classify(reliable, BASE_WEIGHTS)
print('risk_tier value counts (reliable segments):')
print(reliable['risk_tier'].value_counts())
print(f"\nlow_confidence segments (all '{INSUFFICIENT_DATA_TIER}'): {len(low_confidence)}")

risk_tier value counts (reliable segments):
risk_tier
Low risk       14160
Medium risk      386
Name: count, dtype: int64

low_confidence segments (all 'Insufficient data'): 55420


In [3]:
cols = ['segment_id', 'road_class', 'SpeedLimit', 'F85thPercentileSpeed', 'speed_gap',
        'risk_tier', 'speed_safety_score', 'RoadLength_km', 'mapillary_url']
display(reliable.nlargest(10, 'speed_safety_score')[cols])

,segment_id,road_class,SpeedLimit,F85thPercentileSpeed,speed_gap,risk_tier,speed_safety_score,RoadLength_km,mapillary_url
11548,SE/6695,secondary,80.0,40.000000,0.000000,Medium risk,53.3,1.217894,https://www.mapillary.com/app/?lat=21.03999425...
13622,TR/11432,trunk,80.0,104.000000,24.000000,Medium risk,53.2,1.125656,https://www.mapillary.com/app/?lat=18.4029344&...
12926,TR/12368,trunk,60.0,89.222222,29.222222,Medium risk,52.0,2.169816,https://www.mapillary.com/app/?lat=21.0360496&...
14217,TR/11433,trunk,80.0,98.000000,18.000000,Medium risk,51.1,5.172032,https://www.mapillary.com/app/?lat=18.39393775...
12294,SE/5080,secondary,55.0,80.000000,25.000000,Medium risk,50.5,1.029334,https://www.mapillary.com/app/?lat=18.22343805...
13599,TR/11377,trunk,80.0,95.000000,15.000000,Medium risk,50.1,1.130587,https://www.mapillary.com/app/?lat=18.40283550...
13560,TR/13275,trunk,55.0,78.166667,23.166667,Medium risk,49.9,1.439767,https://www.mapillary.com/app/?lat=18.91362550...
14311,MO/216,motorway,60.0,83.200000,23.200000,Medium risk,49.9,6.711048,https://www.mapillary.com/app/?lat=20.0119253&...
14111,MO/141,motorway,55.0,77.800000,22.800000,Medium risk,49.8,1.278202,https://www.mapillary.com/app/?lat=18.97179105...
13642,TR/13274,trunk,55.0,77.500000,22.500000,Medium risk,49.7,1.449865,https://www.mapillary.com/app/?lat=18.91354515...


## Validate against `RankedPercentile`

`RankedPercentile` ranks segments by **travel volume share**, not by safety risk — a busy, well-managed motorway can rank high on traffic but low on risk. A weak or negative correlation is therefore informative about what each metric measures, not necessarily a bug in the score.

In [4]:
valid_corr = reliable[['speed_safety_score', 'RankedPercentile']].dropna()
corr = valid_corr['speed_safety_score'].corr(valid_corr['RankedPercentile'])
print(f'Pearson correlation speed_safety_score vs RankedPercentile: {corr:.4f}')
if corr < 0.3:
    print('WARNING: correlation with RankedPercentile is below 0.3 -- see methodology.md for discussion.')
else:
    print('Correlation is positive and above the 0.3 sanity threshold.')

Pearson correlation speed_safety_score vs RankedPercentile: -0.1955


## Sensitivity analysis

Each weight is varied +-0.10 in 0.05 steps, keeping all four weights summing to 1.0. For every valid combination, the top-20%-by-score segments are compared back to the baseline top 20%.

In [5]:
sens = run_sensitivity_analysis(reliable, BASE_WEIGHTS)
print(f"{sens['n_variants']} valid weight variants tested.")
print(f"Average top-20% overlap with baseline: {sens['average_overlap_pct']:.2f}%")
if sens['robust']:
    print('Score is robust to weight variation')
else:
    print('WARNING: score is sensitive to weight variation')
    print('Segments that flip in/out of the top 20% most often:')
    flippers = reliable.loc[sens['flip_counts'].index[:15], ['segment_id', 'country', 'speed_safety_score']]
    flippers['flip_count'] = sens['flip_counts'].values[:15]
    display(flippers)

84 valid weight variants tested.
Average top-20% overlap with baseline: 90.17%
Score is robust to weight variation


In [6]:
reliable.to_parquet('../data/processed/_reliable_scored.parquet')
low_confidence.to_parquet('../data/processed/_low_confidence_final.parquet')
import json
with open('../data/processed/_sensitivity_summary.json', 'w') as f:
    json.dump({'average_overlap_pct': sens['average_overlap_pct'], 'n_variants': sens['n_variants'],
               'correlation_with_ranked_percentile': corr}, f)
print('Saved intermediate parquet/json for notebook 04.')

Saved intermediate parquet/json for notebook 04.
